# Sweep N_synth - wszystkie 4 modele

Cel: wykres `test_acc(N_synth)` dla wariantu D na wszystkich 4 modelach (resnet18, convnext_tiny, deit_tiny, dinov2_small).

36 nowych runow (4 modele × 3 wartosci N_synth × 3 seedy) + uzycie istniejacych `D_<model>_seed{0,1,2}` runow dla n=120.

In [ ]:
import os
REPO_URL = 'https://github.com/micwuj/DLICV.git'
REPO_DIR = '/content/dlicv'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull

In [ ]:
!pip install -q timm==1.0.11 transformers==4.46.3 huggingface_hub==0.26.2 PyYAML==6.0.2

In [ ]:
from pathlib import Path
PETS = Path('data/raw/oxford-iiit-pet/images')
if not (PETS.exists() and any(PETS.iterdir())):
    !python scripts/download_pets.py

In [ ]:
from huggingface_hub import login, hf_hub_download, whoami
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except (ImportError, Exception):
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('brak HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print('logged in as:', whoami()['name'])

HF_REPO_ID = 'micwuj/dlicv-synth'

In [ ]:
import zipfile, shutil
VARIANT_D = Path('data/synthetic/variant_D')
MANIFEST = VARIANT_D / 'manifest_kept.csv'
sample_png = VARIANT_D / 'Ragdoll' / 'Ragdoll_0000.png'

if not sample_png.exists():
    zip_path = hf_hub_download(repo_id=HF_REPO_ID, repo_type='dataset',
                                filename='variant_D.zip', local_dir='data/synthetic')
    VARIANT_D.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(VARIANT_D)
    nested = VARIANT_D / 'variant_D'
    if nested.exists() and nested.is_dir():
        for d in nested.iterdir():
            tgt = VARIANT_D / d.name
            if tgt.exists(): shutil.rmtree(tgt)
            shutil.move(str(d), str(tgt))
        nested.rmdir()
    print(f'rozpakowane pngs: {sum(1 for _ in VARIANT_D.rglob("*.png"))}')
else:
    print(f'variant_D juz rozpakowane ({sum(1 for _ in VARIANT_D.rglob("*.png"))} pngs).')

hf_hub_download(repo_id=HF_REPO_ID, repo_type='dataset',
                filename='variant_D/manifest_kept.csv', local_dir='data/synthetic')
with open(MANIFEST) as f:
    n_rows = sum(1 for _ in f) - 1
print(f'manifest: {n_rows} rows')
assert n_rows > 2000 and sample_png.exists()

## Sweep: 36 runow

In [ ]:
!python scripts/generate_sweep_configs.py

In [ ]:
# dinov2_small
!python -u scripts/run_all.py --pattern 'D_dinov2_small_n*_seed*.yaml' --overrides configs/colab.yaml --skip-existing

In [ ]:
# resnet18
!python -u scripts/run_all.py --pattern 'D_resnet18_n*_seed*.yaml' --overrides configs/colab.yaml --skip-existing

In [ ]:
# deit_tiny
!python -u scripts/run_all.py --pattern 'D_deit_tiny_n*_seed*.yaml' --overrides configs/colab.yaml --skip-existing

In [ ]:
# convnext_tiny
!python -u scripts/run_all.py --pattern 'D_convnext_tiny_n*_seed*.yaml' --overrides configs/colab.yaml --skip-existing

## Wykres + final dump

In [ ]:
!PYTHONPATH=. python scripts/plot_n_synth_sweep.py

In [ ]:
shutil.make_archive('/content/outputs_sweep', 'zip', '.', 'outputs/D_dinov2_small_n30_seed0')
for run in ['D_dinov2_small_n30_seed1', 'D_dinov2_small_n30_seed2',
            'D_dinov2_small_n60_seed0', 'D_dinov2_small_n60_seed1', 'D_dinov2_small_n60_seed2',
            'D_dinov2_small_n240_seed0', 'D_dinov2_small_n240_seed1', 'D_dinov2_small_n240_seed2']:
    src = Path('outputs') / run
    if src.exists():
        pass
shutil.make_archive('/content/outputs_sweep', 'zip', '.', 'outputs')
files.download('/content/outputs_sweep.zip')